# 10 — Bridge Table (M:N) — DuckDB

Padrão Kimball para relações M:N entre Employee e Territory.

In [1]:
import sys, os
sys.path.insert(0, os.getcwd())
from utils import get_conn, DB_PATH, DATA_DIR

conn = get_conn()
print(f"Conectado: {DB_PATH}")

Conectado: /workspace/pf_northwind/duckdb/northwind_dw.duckdb


In [2]:
# ============================================================
# Criar Bridge Table
# ============================================================
conn.execute("""
    CREATE OR REPLACE TABLE gold.BridgeEmployeeTerritory AS
    SELECT DISTINCT de.EmployeeSK, dt.TerritorySK
    FROM bronze.employee_territories et
    JOIN gold.DimEmployee  de ON de.EmployeeID  = et.EmployeeID
    JOIN gold.DimTerritory dt ON dt.TerritoryID = et.TerritoryID
""")

n = conn.execute("SELECT COUNT(*) AS n FROM gold.BridgeEmployeeTerritory").fetchdf()['n'][0]
print(f"Bridge carregada: {n} linhas")
assert n == 49, f"Esperado 49, got {n}"
print("✓ Bridge OK")


Bridge carregada: 49 linhas
✓ Bridge OK


In [3]:
# ============================================================
# DEMO 1: Receita por território via bridge
# ============================================================
conn.execute("""
    SELECT dt.RegionName, dt.TerritoryDescription,
           COUNT(DISTINCT fs.OrderID) AS OrderCount,
           ROUND(SUM(fs.NetRevenue), 2) AS TotalRevenue
    FROM gold.FactSales fs
    JOIN gold.DimEmployee de ON de.EmployeeSK = fs.EmployeeSK
    JOIN gold.BridgeEmployeeTerritory b ON b.EmployeeSK = de.EmployeeSK
    JOIN gold.DimTerritory dt ON dt.TerritorySK = b.TerritorySK
    GROUP BY dt.RegionName, dt.TerritoryDescription
    ORDER BY TotalRevenue DESC
    LIMIT 10
""").fetchdf()


,RegionName,TerritoryDescription,OrderCount,TotalRevenue
0,Eastern ...,Cary ...,156,232890.85
1,Eastern ...,Greensboro ...,156,232890.85
2,Eastern ...,Rockville ...,156,232890.85
3,Southern ...,Atlanta ...,127,202812.84
4,Southern ...,Orlando ...,127,202812.84
5,Southern ...,Savannah ...,127,202812.84
6,Southern ...,Tampa ...,127,202812.84
7,Eastern ...,Neward ...,123,192107.60
8,Eastern ...,Wilton ...,123,192107.60
9,Eastern ...,Louisville ...,96,166537.75


In [4]:
# ============================================================
# DEMO 2: Empregados com mais territórios
# ============================================================
conn.execute("""
    SELECT de.FullName,
           COUNT(b.TerritorySK) AS QuantidadeTerretorios
    FROM gold.BridgeEmployeeTerritory b
    JOIN gold.DimEmployee de ON de.EmployeeSK = b.EmployeeSK
    GROUP BY de.FullName
    ORDER BY QuantidadeTerretorios DESC
""").fetchdf()


,FullName,QuantidadeTerretorios
0,Robert King,10
1,Andrew Fuller,7
2,Steven Buchanan,7
3,Anne Dodsworth,7
4,Michael Suyama,5
5,Laura Callahan,4
6,Janet Leverling,4
7,Margaret Peacock,3
8,Nancy Davolio,2
